# Phase 1: Simplified Membrane Flux Model for Butane Isomer Separation

## Objective
Build and validate a simplified permeation model for MFI zeolite membranes separating n-butane/i-butane mixtures.

## Reference
Mittal et al. (2016) "A mathematical model for zeolite membrane module performance..." Journal of Membrane Science 520, 434-449.

## Approach
We use a **constant effective permeance** model as a starting point:

$$J_i = \Pi_i \times (p_{i,feed} - p_{i,permeate})$$

Where:
- $J_i$ = molar flux of component i [mol/m²·s]
- $\Pi_i$ = effective permeance of component i [mol/(m²·s·Pa)]
- $p_{i,feed}$ = partial pressure on feed side [Pa]
- $p_{i,permeate}$ = partial pressure on permeate side [Pa]

This lumps together:
- Zeolite layer transport (adsorption + diffusion)
- Support layer resistance
- Concentration polarization effects

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple, Dict

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Physical Constants and Unit Conversions

In [ ]:
# Physical constants
R = 8.314  # Universal gas constant [J/(mol·K)]

# Unit conversions
ATM_TO_PA = 101325  # 1 atm in Pa
BAR_TO_PA = 100000  # 1 bar in Pa

# Component properties
COMPONENTS = {
    'n-butane': {
        'MW': 58.12,  # Molecular weight [g/mol]
        'kinetic_diameter': 0.43e-9,  # [m]
    },
    'i-butane': {
        'MW': 58.12,  # Molecular weight [g/mol]
        'kinetic_diameter': 0.50e-9,  # [m]
    }
}

print("Physical constants and component properties loaded.")
print(f"\nComponent properties:")
for comp, props in COMPONENTS.items():
    print(f"  {comp}: MW = {props['MW']} g/mol, kinetic diameter = {props['kinetic_diameter']*1e9:.2f} nm")

## 2. Experimental Data from Mittal et al. (2016)

### Laboratory Conditions (Figure 3):
- Feed: Equimolar n-butane/i-butane at 1 atm total pressure
- Sweep: Helium at 1 atm (we approximate permeate partial pressures ≈ 0)
- Membrane: 500 nm MFI zeolite layer on 3 mm support
- Temperatures: 300 K, 323 K, 343 K

The experimental data corresponds to membranes with Đ = Đ₀/50 (reduced diffusivity accounting for membrane microstructure vs. ideal crystals).

In [ ]:
# Experimental data extracted from Mittal et al. Figure 3
# These are approximate values read from the bar charts for Đ = Đ₀/50 with 3mm support + conc. polarization

EXPERIMENTAL_DATA = {
    300: {  # Temperature [K]
        'n_butane_flux': 0.012,  # [mol/m²·s] - approximate from Figure 3a
        'separation_factor': 85,  # [-] - approximate from Figure 3b
    },
    323: {
        'n_butane_flux': 0.032,
        'separation_factor': 58,
    },
    343: {
        'n_butane_flux': 0.055,
        'separation_factor': 45,
    }
}

# Laboratory operating conditions
LAB_CONDITIONS = {
    'feed_pressure': 1.0 * ATM_TO_PA,  # [Pa]
    'permeate_pressure': 1.0 * ATM_TO_PA,  # [Pa] - but with He sweep, partial pressures ≈ 0
    'feed_composition': {'n-butane': 0.5, 'i-butane': 0.5},  # Equimolar
}

print("Experimental data from Mittal et al. (2016) Figure 3:")
print("="*60)
print(f"{'T [K]':<10} {'n-C4 Flux [mol/m²·s]':<25} {'Separation Factor':<20}")
print("-"*60)
for T, data in EXPERIMENTAL_DATA.items():
    print(f"{T:<10} {data['n_butane_flux']:<25.4f} {data['separation_factor']:<20}")

## 3. Back-Calculate Effective Permeances from Experimental Data

With the infinite sweep approximation (permeate partial pressure ≈ 0):

$$\Pi_{n-C_4} = \frac{J_{n-C_4}}{p_{n-C_4,feed}}$$

The separation factor for equimolar feed:
$$\alpha = \frac{y_{n-C_4}/y_{i-C_4}}{x_{n-C_4}/x_{i-C_4}} = \frac{J_{n-C_4}}{J_{i-C_4}} = \frac{\Pi_{n-C_4}}{\Pi_{i-C_4}}$$

Therefore:
$$\Pi_{i-C_4} = \frac{\Pi_{n-C_4}}{\alpha}$$

In [ ]:
def calculate_permeances_from_experimental(exp_data: dict, conditions: dict) -> dict:
    """
    Back-calculate effective permeances from experimental flux and separation factor data.
    
    Args:
        exp_data: Dictionary with temperature as key, containing 'n_butane_flux' and 'separation_factor'
        conditions: Operating conditions including 'feed_pressure' and 'feed_composition'
    
    Returns:
        Dictionary with temperature as key, containing permeances for both components
    """
    permeances = {}
    
    # Feed partial pressure of n-butane
    p_nC4_feed = conditions['feed_pressure'] * conditions['feed_composition']['n-butane']
    
    for T, data in exp_data.items():
        # n-butane permeance from flux (assuming p_permeate ≈ 0)
        Pi_nC4 = data['n_butane_flux'] / p_nC4_feed
        
        # i-butane permeance from separation factor
        Pi_iC4 = Pi_nC4 / data['separation_factor']
        
        permeances[T] = {
            'n-butane': Pi_nC4,
            'i-butane': Pi_iC4,
        }
    
    return permeances

# Calculate permeances
calculated_permeances = calculate_permeances_from_experimental(EXPERIMENTAL_DATA, LAB_CONDITIONS)

print("Back-calculated effective permeances:")
print("="*70)
print(f"{'T [K]':<10} {'Π_nC4 [mol/(m²·s·Pa)]':<25} {'Π_iC4 [mol/(m²·s·Pa)]':<25}")
print("-"*70)
for T, perm in calculated_permeances.items():
    print(f"{T:<10} {perm['n-butane']:<25.4e} {perm['i-butane']:<25.4e}")

## 4. Temperature-Dependent Permeance Model

We fit an Arrhenius-type relationship to capture temperature dependence:

$$\Pi_i(T) = \Pi_i^0 \exp\left(-\frac{E_{a,i}}{R}\left(\frac{1}{T} - \frac{1}{T_{ref}}\right)\right)$$

Where:
- $\Pi_i^0$ = permeance at reference temperature $T_{ref}$ [mol/(m²·s·Pa)]
- $E_{a,i}$ = apparent activation energy [J/mol]
- $T_{ref}$ = reference temperature (we'll use 300 K)

In [ ]:
def fit_arrhenius_permeance(temperatures: list, permeances: list, T_ref: float = 300.0) -> Tuple[float, float]:
    """
    Fit Arrhenius parameters to temperature-dependent permeance data.
    
    ln(Π) = ln(Π₀) - (Ea/R) * (1/T - 1/T_ref)
    
    Args:
        temperatures: List of temperatures [K]
        permeances: List of permeances [mol/(m²·s·Pa)]
        T_ref: Reference temperature [K]
    
    Returns:
        Tuple of (Π₀, Ea) - reference permeance and activation energy
    """
    T_arr = np.array(temperatures)
    Pi_arr = np.array(permeances)
    
    # Linear regression: ln(Π) vs (1/T - 1/T_ref)
    x = 1/T_arr - 1/T_ref
    y = np.log(Pi_arr)
    
    # Fit: y = a + b*x where b = -Ea/R
    coeffs = np.polyfit(x, y, 1)
    
    Ea = -coeffs[0] * R  # Activation energy [J/mol]
    Pi_0 = np.exp(coeffs[1])  # Reference permeance
    
    return Pi_0, Ea

# Extract data for fitting
temps = list(calculated_permeances.keys())
perm_nC4 = [calculated_permeances[T]['n-butane'] for T in temps]
perm_iC4 = [calculated_permeances[T]['i-butane'] for T in temps]

# Fit Arrhenius parameters
T_REF = 300.0  # Reference temperature [K]

Pi0_nC4, Ea_nC4 = fit_arrhenius_permeance(temps, perm_nC4, T_REF)
Pi0_iC4, Ea_iC4 = fit_arrhenius_permeance(temps, perm_iC4, T_REF)

print("Arrhenius parameters for temperature-dependent permeance:")
print("="*60)
print(f"Reference temperature: {T_REF} K")
print(f"\nn-Butane:")
print(f"  Π₀ = {Pi0_nC4:.4e} mol/(m²·s·Pa)")
print(f"  Ea = {Ea_nC4/1000:.2f} kJ/mol")
print(f"\ni-Butane:")
print(f"  Π₀ = {Pi0_iC4:.4e} mol/(m²·s·Pa)")
print(f"  Ea = {Ea_iC4/1000:.2f} kJ/mol")

## 5. Membrane Flux Model Class

Now we create a reusable class that encapsulates the simplified membrane model.

In [ ]:
@dataclass
class MembraneParameters:
    """Parameters for the simplified membrane permeance model."""
    Pi0_nC4: float  # Reference permeance for n-butane [mol/(m²·s·Pa)]
    Pi0_iC4: float  # Reference permeance for i-butane [mol/(m²·s·Pa)]
    Ea_nC4: float   # Activation energy for n-butane [J/mol]
    Ea_iC4: float   # Activation energy for i-butane [J/mol]
    T_ref: float    # Reference temperature [K]


class SimplifiedMembraneModel:
    """
    Simplified membrane flux model for butane isomer separation.
    
    Uses constant effective permeance with Arrhenius temperature dependence.
    Assumes infinite sweep (permeate partial pressure ≈ 0) for laboratory conditions.
    """
    
    def __init__(self, params: MembraneParameters):
        self.params = params
        self.R = 8.314  # Gas constant [J/(mol·K)]
    
    def permeance(self, component: str, T: float) -> float:
        """
        Calculate temperature-dependent permeance.
        
        Args:
            component: 'n-butane' or 'i-butane'
            T: Temperature [K]
        
        Returns:
            Permeance [mol/(m²·s·Pa)]
        """
        if component == 'n-butane':
            Pi0 = self.params.Pi0_nC4
            Ea = self.params.Ea_nC4
        elif component == 'i-butane':
            Pi0 = self.params.Pi0_iC4
            Ea = self.params.Ea_iC4
        else:
            raise ValueError(f"Unknown component: {component}")
        
        return Pi0 * np.exp(-Ea/self.R * (1/T - 1/self.params.T_ref))
    
    def flux(self, component: str, T: float, p_feed: float, p_permeate: float = 0.0) -> float:
        """
        Calculate molar flux through membrane.
        
        Args:
            component: 'n-butane' or 'i-butane'
            T: Temperature [K]
            p_feed: Partial pressure on feed side [Pa]
            p_permeate: Partial pressure on permeate side [Pa]
        
        Returns:
            Molar flux [mol/(m²·s)]
        """
        Pi = self.permeance(component, T)
        return Pi * (p_feed - p_permeate)
    
    def separation_factor(self, T: float, x_nC4: float = 0.5, 
                          p_total_feed: float = 101325, 
                          p_permeate_nC4: float = 0.0,
                          p_permeate_iC4: float = 0.0) -> float:
        """
        Calculate separation factor.
        
        α = (y_nC4/y_iC4) / (x_nC4/x_iC4)
        
        For infinite sweep (p_permeate ≈ 0), this simplifies to:
        α = Π_nC4 / Π_iC4 (independent of composition)
        
        Args:
            T: Temperature [K]
            x_nC4: Mole fraction of n-butane in feed
            p_total_feed: Total pressure on feed side [Pa]
            p_permeate_nC4: Partial pressure of n-butane on permeate side [Pa]
            p_permeate_iC4: Partial pressure of i-butane on permeate side [Pa]
        
        Returns:
            Separation factor [-]
        """
        x_iC4 = 1 - x_nC4
        
        # Feed partial pressures
        p_feed_nC4 = x_nC4 * p_total_feed
        p_feed_iC4 = x_iC4 * p_total_feed
        
        # Fluxes
        J_nC4 = self.flux('n-butane', T, p_feed_nC4, p_permeate_nC4)
        J_iC4 = self.flux('i-butane', T, p_feed_iC4, p_permeate_iC4)
        
        # Permeate composition
        J_total = J_nC4 + J_iC4
        y_nC4 = J_nC4 / J_total
        y_iC4 = J_iC4 / J_total
        
        # Separation factor
        alpha = (y_nC4 / y_iC4) / (x_nC4 / x_iC4)
        
        return alpha
    
    def ideal_selectivity(self, T: float) -> float:
        """
        Calculate ideal selectivity (ratio of permeances).
        
        For infinite sweep, this equals the separation factor.
        
        Args:
            T: Temperature [K]
        
        Returns:
            Ideal selectivity [-]
        """
        Pi_nC4 = self.permeance('n-butane', T)
        Pi_iC4 = self.permeance('i-butane', T)
        return Pi_nC4 / Pi_iC4
    
    def simulate_well_mixed(self, T: float, p_total_feed: float, x_nC4_feed: float,
                            p_permeate_nC4: float = 0.0, p_permeate_iC4: float = 0.0) -> Dict:
        """
        Simulate a well-mixed membrane element.
        
        Args:
            T: Temperature [K]
            p_total_feed: Total pressure on feed side [Pa]
            x_nC4_feed: Mole fraction of n-butane in feed
            p_permeate_nC4: Partial pressure of n-butane on permeate side [Pa]
            p_permeate_iC4: Partial pressure of i-butane on permeate side [Pa]
        
        Returns:
            Dictionary with simulation results
        """
        x_iC4_feed = 1 - x_nC4_feed
        
        # Feed partial pressures
        p_feed_nC4 = x_nC4_feed * p_total_feed
        p_feed_iC4 = x_iC4_feed * p_total_feed
        
        # Permeances
        Pi_nC4 = self.permeance('n-butane', T)
        Pi_iC4 = self.permeance('i-butane', T)
        
        # Fluxes
        J_nC4 = self.flux('n-butane', T, p_feed_nC4, p_permeate_nC4)
        J_iC4 = self.flux('i-butane', T, p_feed_iC4, p_permeate_iC4)
        J_total = J_nC4 + J_iC4
        
        # Permeate composition
        y_nC4 = J_nC4 / J_total if J_total > 0 else 0
        y_iC4 = J_iC4 / J_total if J_total > 0 else 0
        
        # Separation factor
        alpha = (y_nC4 / y_iC4) / (x_nC4_feed / x_iC4_feed) if (y_iC4 > 0 and x_iC4_feed > 0) else 0
        
        return {
            'T': T,
            'p_total_feed': p_total_feed,
            'x_nC4_feed': x_nC4_feed,
            'x_iC4_feed': x_iC4_feed,
            'Pi_nC4': Pi_nC4,
            'Pi_iC4': Pi_iC4,
            'J_nC4': J_nC4,
            'J_iC4': J_iC4,
            'J_total': J_total,
            'y_nC4': y_nC4,
            'y_iC4': y_iC4,
            'separation_factor': alpha,
        }


# Create model instance with fitted parameters
membrane_params = MembraneParameters(
    Pi0_nC4=Pi0_nC4,
    Pi0_iC4=Pi0_iC4,
    Ea_nC4=Ea_nC4,
    Ea_iC4=Ea_iC4,
    T_ref=T_REF
)

model = SimplifiedMembraneModel(membrane_params)

print("SimplifiedMembraneModel created successfully.")
print(f"\nModel parameters:")
print(f"  Π₀_nC4 = {membrane_params.Pi0_nC4:.4e} mol/(m²·s·Pa)")
print(f"  Π₀_iC4 = {membrane_params.Pi0_iC4:.4e} mol/(m²·s·Pa)")
print(f"  Ea_nC4 = {membrane_params.Ea_nC4/1000:.2f} kJ/mol")
print(f"  Ea_iC4 = {membrane_params.Ea_iC4/1000:.2f} kJ/mol")

## 6. Model Validation Against Experimental Data

Let's compare our model predictions with the experimental data from Mittal et al.

In [ ]:
# Validate model against experimental data
print("Model Validation:")
print("="*90)
print(f"{'T [K]':<8} {'J_nC4 exp':<15} {'J_nC4 model':<15} {'Error %':<10} {'α exp':<10} {'α model':<10} {'Error %':<10}")
print("-"*90)

validation_results = []

for T, exp_data in EXPERIMENTAL_DATA.items():
    # Experimental values
    J_exp = exp_data['n_butane_flux']
    alpha_exp = exp_data['separation_factor']
    
    # Model predictions (lab conditions: 1 atm, equimolar feed, infinite sweep)
    results = model.simulate_well_mixed(
        T=T,
        p_total_feed=1.0 * ATM_TO_PA,
        x_nC4_feed=0.5,
        p_permeate_nC4=0.0,
        p_permeate_iC4=0.0
    )
    
    J_model = results['J_nC4']
    alpha_model = results['separation_factor']
    
    # Errors
    J_error = 100 * (J_model - J_exp) / J_exp
    alpha_error = 100 * (alpha_model - alpha_exp) / alpha_exp
    
    validation_results.append({
        'T': T,
        'J_exp': J_exp,
        'J_model': J_model,
        'J_error': J_error,
        'alpha_exp': alpha_exp,
        'alpha_model': alpha_model,
        'alpha_error': alpha_error,
    })
    
    print(f"{T:<8} {J_exp:<15.4f} {J_model:<15.4f} {J_error:<10.2f} {alpha_exp:<10} {alpha_model:<10.1f} {alpha_error:<10.2f}")

print("\nNote: Small errors are expected as we fit an Arrhenius model to 3 data points.")

## 7. Visualization

In [ ]:
# Create validation plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Temperature range for model curves
T_range = np.linspace(290, 360, 100)

# Experimental data points
T_exp = list(EXPERIMENTAL_DATA.keys())
J_exp = [EXPERIMENTAL_DATA[T]['n_butane_flux'] for T in T_exp]
alpha_exp = [EXPERIMENTAL_DATA[T]['separation_factor'] for T in T_exp]

# Model predictions over temperature range
J_model_range = []
alpha_model_range = []
Pi_nC4_range = []
Pi_iC4_range = []

for T in T_range:
    results = model.simulate_well_mixed(
        T=T,
        p_total_feed=1.0 * ATM_TO_PA,
        x_nC4_feed=0.5
    )
    J_model_range.append(results['J_nC4'])
    alpha_model_range.append(results['separation_factor'])
    Pi_nC4_range.append(results['Pi_nC4'])
    Pi_iC4_range.append(results['Pi_iC4'])

# Plot 1: n-Butane Flux vs Temperature
ax1 = axes[0, 0]
ax1.semilogy(T_range, J_model_range, 'b-', linewidth=2, label='Model')
ax1.semilogy(T_exp, J_exp, 'ro', markersize=10, label='Experimental (Mittal et al.)')
ax1.set_xlabel('Temperature [K]')
ax1.set_ylabel('n-Butane Flux [mol/(m²·s)]')
ax1.set_title('n-Butane Flux vs Temperature')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Separation Factor vs Temperature
ax2 = axes[0, 1]
ax2.plot(T_range, alpha_model_range, 'b-', linewidth=2, label='Model')
ax2.plot(T_exp, alpha_exp, 'ro', markersize=10, label='Experimental (Mittal et al.)')
ax2.set_xlabel('Temperature [K]')
ax2.set_ylabel('Separation Factor [-]')
ax2.set_title('Separation Factor vs Temperature')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Permeances vs Temperature (Arrhenius plot)
ax3 = axes[1, 0]
ax3.semilogy(1000/T_range, Pi_nC4_range, 'b-', linewidth=2, label='n-Butane')
ax3.semilogy(1000/T_range, Pi_iC4_range, 'r-', linewidth=2, label='i-Butane')
ax3.set_xlabel('1000/T [1/K]')
ax3.set_ylabel('Permeance [mol/(m²·s·Pa)]')
ax3.set_title('Arrhenius Plot of Permeances')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.invert_xaxis()  # Higher T on left

# Plot 4: Parity plot
ax4 = axes[1, 1]
J_model_points = [model.simulate_well_mixed(T, 1.0*ATM_TO_PA, 0.5)['J_nC4'] for T in T_exp]
ax4.plot([0, 0.08], [0, 0.08], 'k--', linewidth=1, label='Parity')
ax4.scatter(J_exp, J_model_points, c='blue', s=100, zorder=5)
for i, T in enumerate(T_exp):
    ax4.annotate(f'{T} K', (J_exp[i], J_model_points[i]), 
                 textcoords="offset points", xytext=(5, 5), fontsize=10)
ax4.set_xlabel('Experimental Flux [mol/(m²·s)]')
ax4.set_ylabel('Model Flux [mol/(m²·s)]')
ax4.set_title('Parity Plot: Model vs Experimental')
ax4.set_xlim([0, 0.08])
ax4.set_ylim([0, 0.08])
ax4.set_aspect('equal')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/phase1_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved to '../figures/phase1_validation.png'")

## 8. Sensitivity Analysis: Effect of Operating Conditions

Let's explore how flux and separation factor vary with:
1. Feed pressure
2. Feed composition
3. Temperature

In [ ]:
# Sensitivity analysis
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Reference conditions (industrial-like)
T_ref_sens = 343  # K
P_ref_sens = 8.81 * ATM_TO_PA  # Pa (Mittal's industrial conditions)
x_ref_sens = 0.696  # Mittal's industrial feed composition

# 1. Effect of feed pressure (at constant T and composition)
P_range = np.linspace(1, 20, 50) * ATM_TO_PA
J_vs_P = []
alpha_vs_P = []

for P in P_range:
    results = model.simulate_well_mixed(T_ref_sens, P, x_ref_sens)
    J_vs_P.append(results['J_nC4'])
    alpha_vs_P.append(results['separation_factor'])

axes[0, 0].plot(P_range/ATM_TO_PA, J_vs_P, 'b-', linewidth=2)
axes[0, 0].set_xlabel('Feed Pressure [atm]')
axes[0, 0].set_ylabel('n-Butane Flux [mol/(m²·s)]')
axes[0, 0].set_title(f'Flux vs Pressure (T={T_ref_sens}K, x_nC4={x_ref_sens})')
axes[0, 0].grid(True, alpha=0.3)

axes[1, 0].plot(P_range/ATM_TO_PA, alpha_vs_P, 'r-', linewidth=2)
axes[1, 0].set_xlabel('Feed Pressure [atm]')
axes[1, 0].set_ylabel('Separation Factor [-]')
axes[1, 0].set_title('Separation Factor vs Pressure')
axes[1, 0].grid(True, alpha=0.3)

# 2. Effect of feed composition (at constant T and P)
x_range = np.linspace(0.1, 0.9, 50)
J_vs_x = []
alpha_vs_x = []

for x in x_range:
    results = model.simulate_well_mixed(T_ref_sens, P_ref_sens, x)
    J_vs_x.append(results['J_nC4'])
    alpha_vs_x.append(results['separation_factor'])

axes[0, 1].plot(x_range * 100, J_vs_x, 'b-', linewidth=2)
axes[0, 1].set_xlabel('n-Butane Feed Composition [mol%]')
axes[0, 1].set_ylabel('n-Butane Flux [mol/(m²·s)]')
axes[0, 1].set_title(f'Flux vs Composition (T={T_ref_sens}K, P={P_ref_sens/ATM_TO_PA:.1f}atm)')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 1].plot(x_range * 100, alpha_vs_x, 'r-', linewidth=2)
axes[1, 1].set_xlabel('n-Butane Feed Composition [mol%]')
axes[1, 1].set_ylabel('Separation Factor [-]')
axes[1, 1].set_title('Separation Factor vs Composition')
axes[1, 1].grid(True, alpha=0.3)

# 3. Effect of temperature (at constant P and composition)
T_sens_range = np.linspace(290, 370, 50)
J_vs_T = []
alpha_vs_T = []

for T in T_sens_range:
    results = model.simulate_well_mixed(T, P_ref_sens, x_ref_sens)
    J_vs_T.append(results['J_nC4'])
    alpha_vs_T.append(results['separation_factor'])

axes[0, 2].plot(T_sens_range, J_vs_T, 'b-', linewidth=2)
axes[0, 2].set_xlabel('Temperature [K]')
axes[0, 2].set_ylabel('n-Butane Flux [mol/(m²·s)]')
axes[0, 2].set_title(f'Flux vs Temperature (P={P_ref_sens/ATM_TO_PA:.1f}atm, x_nC4={x_ref_sens})')
axes[0, 2].grid(True, alpha=0.3)

axes[1, 2].plot(T_sens_range, alpha_vs_T, 'r-', linewidth=2)
axes[1, 2].set_xlabel('Temperature [K]')
axes[1, 2].set_ylabel('Separation Factor [-]')
axes[1, 2].set_title('Separation Factor vs Temperature')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/phase1_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved to '../figures/phase1_sensitivity.png'")

## 9. Key Observations and Limitations

### Observations from the Simplified Model:

1. **Flux increases with temperature** - Higher thermal energy increases diffusivity
2. **Separation factor decreases with temperature** - i-butane diffusivity increases faster than n-butane
3. **Flux is linear with feed pressure** - Expected from our linear driving force assumption
4. **Separation factor is constant with pressure** - This is a limitation of our simplified model!

### Limitations of the Simplified Model:

1. **No concentration polarization** - Real membranes show reduced performance at high flux
2. **No support resistance variation** - Support resistance becomes significant at high pressures
3. **No loading-dependent permeance** - RAST predicts non-linear adsorption effects
4. **Infinite sweep assumption** - Real permeate has finite partial pressures

### When This Model is Adequate:

- Quick screening calculations
- Process flow sheet development (Phase 2)
- Order-of-magnitude area estimation
- Understanding parameter sensitivities

### When You Need the Full Model:

- Detailed membrane design
- Accurate flux predictions at high pressures
- Optimization studies
- Comparison with experimental data at varied conditions

## 10. Export Model Parameters for Phase 2

Save the fitted parameters for use in the process simulation.

In [ ]:
import json

# Export parameters to JSON
model_params_export = {
    'model_type': 'simplified_constant_permeance',
    'reference': 'Mittal et al. (2016) J. Membrane Sci. 520, 434-449',
    'fit_conditions': {
        'feed_pressure_Pa': 101325,
        'feed_composition': 'equimolar n-butane/i-butane',
        'sweep': 'helium at 1 atm (infinite sweep approximation)',
        'membrane': '500 nm MFI zeolite on 3 mm support',
    },
    'parameters': {
        'T_ref_K': T_REF,
        'Pi0_nC4_mol_m2_s_Pa': Pi0_nC4,
        'Pi0_iC4_mol_m2_s_Pa': Pi0_iC4,
        'Ea_nC4_J_mol': Ea_nC4,
        'Ea_iC4_J_mol': Ea_iC4,
    },
    'validation': {
        'temperatures_K': T_exp,
        'experimental_flux_mol_m2_s': J_exp,
        'experimental_separation_factor': alpha_exp,
    }
}

# Save to file
with open('../data/membrane_model_params.json', 'w') as f:
    json.dump(model_params_export, f, indent=2)

print("Model parameters exported to '../data/membrane_model_params.json'")
print("\nExported parameters:")
print(json.dumps(model_params_export, indent=2))

## 11. Next Steps (Phase 2 Preview)

In Phase 2, we will:

1. **Implement plug flow model** along the membrane length
   - Discretize membrane into increments
   - Track composition changes along the length
   - Include pressure drop calculations

2. **Add multi-stage configurations**
   - Single stage membrane
   - Two-stage with recycle
   - Series hybrid with distillation

3. **Calculate membrane area requirements**
   - For target purity and recovery
   - Sensitivity to operating conditions

The `SimplifiedMembraneModel` class we built here will be the foundation for these process-level simulations.

In [ ]:
# Quick preview: what area would we need for a given separation?
# This is a very rough estimate assuming well-mixed behavior

def estimate_membrane_area_simple(model, T, P_feed, x_nC4_feed, F_feed, recovery_nC4):
    """
    Rough estimate of membrane area for a target recovery.
    Assumes well-mixed behavior (very approximate!).
    
    Args:
        model: SimplifiedMembraneModel instance
        T: Temperature [K]
        P_feed: Feed pressure [Pa]
        x_nC4_feed: n-butane mole fraction in feed
        F_feed: Total feed flow rate [mol/s]
        recovery_nC4: Target n-butane recovery (fraction permeated)
    
    Returns:
        Approximate membrane area [m²]
    """
    # n-butane to be permeated
    F_nC4_feed = F_feed * x_nC4_feed
    F_nC4_permeate = F_nC4_feed * recovery_nC4
    
    # Average flux (using feed conditions - this is approximate!)
    results = model.simulate_well_mixed(T, P_feed, x_nC4_feed)
    J_nC4 = results['J_nC4']
    
    # Area estimate
    Area = F_nC4_permeate / J_nC4
    
    return Area

# Example: Mittal's industrial conditions
T_ind = 343  # K
P_ind = 8.81 * ATM_TO_PA  # Pa
x_nC4_ind = 0.696
F_ind = 121.8  # mol/s (from Mittal Table 2)
recovery_target = 0.40  # 40% stage cut

Area_estimate = estimate_membrane_area_simple(model, T_ind, P_ind, x_nC4_ind, F_ind, recovery_target)

print("Rough Membrane Area Estimate (Industrial Conditions):")
print("="*50)
print(f"Temperature: {T_ind} K")
print(f"Feed pressure: {P_ind/ATM_TO_PA:.2f} atm")
print(f"Feed composition: {x_nC4_ind*100:.1f}% n-butane")
print(f"Feed flow rate: {F_ind} mol/s")
print(f"Target recovery: {recovery_target*100:.0f}%")
print(f"\nEstimated membrane area: {Area_estimate:.0f} m²")
print(f"\nNote: Mittal reports ~300 m² for φ=0.4 with improved membranes (Đ=Đ₀/5).")
print(f"Our estimate uses current membranes (Đ=Đ₀/50), so larger area is expected.")